# K-Nearest Neighbors (KNN) Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Crucial for KNN)

**Important:** KNN is distance-based and requires feature scaling for optimal performance. Features must be scaled to similar ranges.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## K-Nearest Neighbors (KNN) Regression Formula and Concepts

**KNN Regression Concept:**

KNN regression predicts the value for a new data point by averaging the values of its k nearest neighbors in the training set.

**Prediction Formula:**

$$\hat{y} = \frac{1}{k} \sum_{i=1}^{k} y_i$$

Where:
- **ŷ** = predicted value for the new point
- **k** = number of nearest neighbors
- **yᵢ** = target value of the i-th nearest neighbor

**Distance Metrics:**

The most common distance metric is Euclidean distance:

$$d(x, x') = \sqrt{\sum_{j=1}^{p} (x_j - x'_j)^2}$$

Other distance metrics:
- **Manhattan Distance**: $d(x, x') = \sum_{j=1}^{p} |x_j - x'_j|$
- **Minkowski Distance**: $d(x, x') = (\sum_{j=1}^{p} |x_j - x'_j|^p)^{1/p}$

**Key Hyperparameters:**
- **n_neighbors (k)**: Number of neighbors to consider
  - Small k: More sensitive to noise (overfitting)
  - Large k: Smoother predictions (underfitting)
- **weights**: How to weight neighbors' contributions
  - 'uniform': All neighbors contribute equally
  - 'distance': Closer neighbors have more influence
- **metric**: Distance metric to use (default='minkowski' with p=2 for Euclidean)
- **algorithm**: Algorithm used to compute nearest neighbors ('auto', 'ball_tree', 'kd_tree', 'brute')

**Advantages:**
- **Simple to understand**: Intuitive algorithm
- **No training phase**: Lazy learning (stores training data)
- **Non-parametric**: Makes no assumptions about data distribution
- **Versatile**: Can handle both classification and regression

**Disadvantages:**
- **Computationally expensive**: Needs to calculate distances to all points
- **Memory intensive**: Stores entire training dataset
- **Slow prediction**: Especially with large datasets
- **Sensitive to outliers**: Can be affected by noisy data
- **Requires feature scaling**: Distance-based algorithm

**Key Considerations:**
- **Curse of dimensionality**: Performance degrades with many features
- **Choice of k**: Critical for performance (use cross-validation)
- **Feature scaling**: Essential for distance-based algorithms

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train KNN with Default Parameters (k=5)

In [ ]:
knn = KNeighborsRegressor(n_neighbors=5)

knn.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = knn.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "KNN Regression (k=5)"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Hyperparameter Tuning: Finding Optimal k

The choice of k is critical for KNN performance. Let's test different values of k.

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21, 31, 51]

results = []

for k in k_values:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'k': k,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Visualize Performance vs k

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(results_df['k'], results_df['R²'], 'o-')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('R² Score')
plt.title('KNN Performance vs k')
plt.grid(True, alpha=0.3)
plt.show()

## Compare Uniform vs Distance Weights

Let's compare uniform weighting (all neighbors equal) with distance weighting (closer neighbors have more influence).

In [ ]:
weights_options = ['uniform', 'distance']

results = []

for weight in weights_options:
    model = KNeighborsRegressor(n_neighbors=5, weights=weight)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'Weights': weight,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Train Optimized KNN Model

Based on the results, let's train a KNN model with the optimal parameters.

In [ ]:
# Using k=7 and distance weights based on our analysis
knn_optimized = KNeighborsRegressor(n_neighbors=7, weights='distance')
knn_optimized.fit(X_train, y_train)
y_pred_opt = knn_optimized.predict(X_test)

r2_opt = r2_score(y_test, y_pred_opt)
print(f"Optimized KNN R²: {r2_opt:.4f}")

## Summary

KNN Regression provides:
- **Simple and intuitive**: Easy to understand and implement
- **Non-parametric**: Makes no assumptions about data distribution
- **Lazy learning**: No training phase, just stores data
- **Versatile**: Works for both classification and regression

**Key considerations:**
- **Feature scaling is crucial**: Distance-based algorithm requires scaled features
- **Choice of k is critical**: Use cross-validation to find optimal k
- **Computationally expensive**: Slow for large datasets
- **Memory intensive**: Stores entire training dataset
- **Curse of dimensionality**: Performance degrades with many features

**Best practices:**
- Always scale features before training
- Use cross-validation to tune k
- Consider distance weighting for better performance
- Use efficient algorithms (KD-tree, Ball-tree) for large datasets
- Consider alternative algorithms for very large datasets